# 📓 Notebook 02 — SMC Indicators อธิบายละเอียด

## ในบทนี้คุณจะได้เรียนรู้:
- SMC Indicators แต่ละตัวหมายความว่าอะไร
- อ่านข้อมูลจาก DataFrame ที่คำนวณได้
- ตีความสัญญาณ Bullish/Bearish

| Indicator | ย่อมาจาก | ความหมาย |
|-----------|----------|----------|
| **FVG** | Fair Value Gap | ช่องราคาขาดสภาพคล่อง |
| **BOS** | Break of Structure | ยืนยัน trend |
| **CHOCH** | Change of Character | สัญญาณ reversal |
| **OB** | Order Block | โซน demand/supply |
| **Liquidity** | Liquidity Pool | จุดรวม Stop Loss |

In [ ]:
# Setup — ใช้โค้ดจาก Notebook 01
import pandas as pd, numpy as np, os
import smartmoneyconcepts as smc

# ดึงข้อมูล (copy จาก NB01)
def get_mt5_data(symbol='XAUUSD', timeframe_str='15m', count=500):
    try:
        import MetaTrader5 as mt5
        if not mt5.initialize(): return None
        tf_map = {'1m': mt5.TIMEFRAME_M1, '5m': mt5.TIMEFRAME_M5, '15m': mt5.TIMEFRAME_M15,
                  '1h': mt5.TIMEFRAME_H1, '4h': mt5.TIMEFRAME_H4, '1d': mt5.TIMEFRAME_D1, '1w': mt5.TIMEFRAME_W1}
        rates = mt5.copy_rates_from_pos(symbol, tf_map.get(timeframe_str, mt5.TIMEFRAME_M15), 0, count)
        mt5.shutdown()
        if rates is None: return None
        df = pd.DataFrame(rates)
        df['date'] = pd.to_datetime(df['time'], unit='s')
        df = df.rename(columns={'tick_volume':'volume'})
        return df[['date','open','high','low','close','volume']].copy()
    except: return None

def get_data(timeframe='15m', count=200):
    df = get_mt5_data(timeframe_str=timeframe, count=count)
    if df is not None: return df
    if os.path.exists('sample_xauusd.csv'):
        return pd.read_csv('sample_xauusd.csv', parse_dates=['date']).tail(count).reset_index(drop=True)
    np.random.seed(42); base=2650; prices=base+np.cumsum(np.random.randn(count)*2.5)
    data=[{'date':pd.Timestamp('2025-01-01')+pd.Timedelta(minutes=15*i),'open':round(p+np.random.randn()*1.5,2),
           'high':round(max(p+np.random.randn()*1.5,p)+abs(np.random.randn())*3,2),
           'low':round(min(p+np.random.randn()*1.5,p)-abs(np.random.randn())*3,2),
           'close':round(p,2),'volume':int(abs(np.random.randn())*1000+500)} for i,p in enumerate(prices)]
    return pd.DataFrame(data)

df = get_data('15m', 200)
swing = smc.smc.swing_highs_lows(df, swing_length=15)
fvg = smc.smc.fvg(df, join_consecutive=False)
bos_choch = smc.smc.bos_choch(df, swing, close_break=True)
ob = smc.smc.ob(df, swing, close_mitigation=False)
liquidity = smc.smc.liquidity(df, swing, range_percent=0.02)
print(f'✅ ข้อมูลพร้อม: {len(df)} แท่ง')

## 🔍 FVG — Fair Value Gap

เกิดเมื่อแท่งเทียนเคลื่อนที่เร็วจนมี "ช่อง" ราคาที่ไม่มีการซื้อขาย

- **Bullish FVG** (FVG=1): ช่อง demand → ราคามักลงมาเติมแล้วเด้งขึ้น
- **Bearish FVG** (FVG=-1): ช่อง supply → ราคามักขึ้นไปเติมแล้วกลับลง
- **MitigatedIndex**: ถ้ามีค่า = ถูกเติมแล้ว (ไม่ active)

In [ ]:
# แสดง FVG ทั้งหมด
fvg_active = fvg[(fvg['FVG'].notna()) & (fvg['FVG'] != 0)].copy()
fvg_active['Type'] = fvg_active['FVG'].map({1: '📈 Bullish', -1: '📉 Bearish'})
fvg_active['Status'] = fvg_active['MitigatedIndex'].apply(lambda x: '❌ Mitigated' if pd.notna(x) else '✅ Active')

active_count = (fvg_active['MitigatedIndex'].isna()).sum()
print(f'=== Fair Value Gap (FVG) ===')
print(f'ทั้งหมด: {len(fvg_active)} gaps | Active: {active_count}')
print()

for _, row in fvg_active.tail(6).iterrows():
    print(f"  {row['Type']} FVG: {row['Bottom']:.2f} — {row['Top']:.2f} [{row['Status']}]")

## 🏛️ BOS & CHOCH

- **BOS** (Break of Structure) = ราคาทะลุ High/Low เก่า → trend ยังดำเนินต่อ
- **CHOCH** (Change of Character) = ราคากลับทิศ → trend กำลังเปลี่ยน

In [ ]:
bos_points = bos_choch[bos_choch['BOS'].notna()]
choch_points = bos_choch[bos_choch['CHOCH'].notna()]

print('=== Break of Structure (BOS) ===')
print(f'พบ BOS: {len(bos_points)} ครั้ง | 3 ล่าสุด:')
for _, row in bos_points.tail(3).iterrows():
    d = '📈 Bullish' if row['BOS']==1 else '📉 Bearish'
    print(f"  {d} BOS ที่ {row['Level']:.2f}")

print(f'\n=== Change of Character (CHOCH) ===')
print(f'พบ CHOCH: {len(choch_points)} ครั้ง | 3 ล่าสุด:')
for _, row in choch_points.tail(3).iterrows():
    d = '📈 Bullish' if row['CHOCH']==1 else '📉 Bearish'
    print(f"  {d} CHOCH ที่ {row['Level']:.2f}")

if len(bos_points)>0:
    last = bos_points.iloc[-1]
    print(f"\n🔑 Trend ปัจจุบัน: {'Bullish' if last['BOS']==1 else 'Bearish'} (จาก BOS ล่าสุด)")

## 🧱 Order Blocks (OB)

แท่งเทียนสุดท้ายก่อน BOS/CHOCH — โซนที่ Smart Money เข้าซื้อ/ขาย

In [ ]:
ob_active = ob[(ob['OB'].notna()) & (ob['OB'] != 0)].copy()
bull_ob = ob_active[(ob_active['OB']==1) & ob_active['MitigatedIndex'].isna()]
bear_ob = ob_active[(ob_active['OB']==-1) & ob_active['MitigatedIndex'].isna()]

print('=== Order Blocks ===')
print(f'🟢 Bullish OB (Active): {len(bull_ob)}')
for _, row in bull_ob.tail(3).iterrows():
    print(f"  Demand Zone: {row['Bottom']:.2f} — {row['Top']:.2f}")

print(f'\n🔴 Bearish OB (Active): {len(bear_ob)}')
for _, row in bear_ob.tail(3).iterrows():
    print(f"  Supply Zone: {row['Bottom']:.2f} — {row['Top']:.2f}")

## 💧 Liquidity Pools

จุดที่มี Stop Loss รวมกัน — เป้าหมายที่ Smart Money ล่า

In [ ]:
liq_active = liquidity[(liquidity['Liquidity'].notna()) & (liquidity['Liquidity'] != 0)]
bsl = liq_active[(liq_active['Liquidity']==1) & liq_active['Swept'].isna()]
ssl = liq_active[(liq_active['Liquidity']==-1) & liq_active['Swept'].isna()]

print('=== Liquidity ===')
print(f'🎯 BSL (Buy-side, เป้าหมายขาขึ้น): {len(bsl)}')
for _, row in bsl.tail(3).iterrows():
    print(f"  {row['Level']:.2f}")
print(f'\n🎯 SSL (Sell-side, เป้าหมายขาลง): {len(ssl)}')
for _, row in ssl.tail(3).iterrows():
    print(f"  {row['Level']:.2f}")

## ✅ สรุป Notebook 02

คุณเข้าใจ SMC Indicators ตัวหลักทั้ง 5 แล้ว:
1. ✅ **FVG** = ช่องว่างราคา → POI (Point of Interest)
2. ✅ **BOS** = ยืนยัน trend → ตามเทรนด์
3. ✅ **CHOCH** = เปลี่ยนเทรนด์ → ระวัง reversal
4. ✅ **OB** = โซน entry ที่ดีที่สุด
5. ✅ **Liquidity** = เป้าหมายของ Smart Money

---
**➡️ ต่อไป**: `03_llm_analysis.ipynb` — ส่งข้อมูลเหล่านี้ให้ GPT-4o วิเคราะห์